<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/Notebooks/Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1b: Classical Random Forest Training

In [1]:
!pip install category_encoders

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')
import dataPrep

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def splitBetweenTrainAndDev(df):
  train,dev = train_test_split(df,test_size=0.2,random_state=51)
  return train,dev

In [4]:
def trainAndTestModel(model, train, dev):
  model.fit(train.drop(columns=['isSubscribed']), train['isSubscribed'])
  train_predictions = model.predict(train.drop(columns=['isSubscribed']))
  dev_predictions = model.predict(dev.drop(columns=['isSubscribed']))
  trainScore = f1_score(train['isSubscribed'], train_predictions)
  devScore = f1_score(dev['isSubscribed'], dev_predictions)
  return trainScore, devScore

In [5]:
def createModel(criterion, nEstimators, maxDepth, maxFeatures):
  return rf(criterion=criterion, n_estimators=nEstimators, max_depth=maxDepth, max_features=maxFeatures)

In [6]:
def createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary):
  return pd.DataFrame(scoreVsHyperparameterDictionary).sort_values(by=['devScore','trainScore'],ascending=False)

In [7]:
def appendToHyperparameterDictionary(scoreVsHyperparameterDictionary, criterion, nEstimators, maxDepth, maxFeatures, trainScore, devScore):
  scoreVsHyperparameterDictionary['criterion'].append(criterion)
  scoreVsHyperparameterDictionary['nEstimators'].append(nEstimators)
  scoreVsHyperparameterDictionary['maxDepth'].append(maxDepth)
  scoreVsHyperparameterDictionary['maxFeatures'].append(maxFeatures)
  scoreVsHyperparameterDictionary['trainScore'].append(trainScore)
  scoreVsHyperparameterDictionary['devScore'].append(devScore)
  return scoreVsHyperparameterDictionary

In [8]:
def modelExistsInDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return exists(fullName)

In [9]:
def saveModelToDrive(data,filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  pickle.dump(data, open(fullName, 'wb'))

In [10]:
def retrieveModelFromDrive(filename):
  # Drive mounts to /content/drive/My Drive/
  fullName = f"/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Models/{filename}"
  return pickle.load(open(fullName, 'rb'))

In [11]:
def adjustRange(dfColumn):
  if dfColumn.min == dfColumn.max:
    return np.arange(2,2*dfColumn.max(),1)
  return np.arange(dfColumn.min(),dfColumn.max(),1)

In [12]:
def main():
  trainSet = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
  train,dev = splitBetweenTrainAndDev(trainSet)
  nColumns = len(train.columns)
  bestDevScore = 0

  possibleCriterion = ["gini","entropy","log_loss"]
  possibleNEstimators = np.arange(5, 100, 1)
  possibleMaxDepth = np.arange(2,nColumns,1)
  possibleMaxFeatures = np.arange(2,nColumns,1)

  scoreVsHyperparameterDictionary = {
      "criterion":[],
      "nEstimators":[],
      "maxDepth":[],
      "maxFeatures":[],
      "trainScore":[],
      "devScore":[]
  }

  for iteration in range(50):
    print(f"iteration: {iteration+1} of 50")
    criterion = np.random.choice(possibleCriterion)
    nEstimators = np.random.choice(possibleNEstimators)
    maxDepth = np.random.choice(possibleMaxDepth)
    maxFeatures = np.random.choice(possibleMaxFeatures)
    model = createModel(criterion, nEstimators, maxDepth, maxFeatures)
    trainScore, devScore = trainAndTestModel(model, train, dev)
    if (trainScore > .8) and (devScore > bestDevScore):
      bestDevScore = devScore
      saveModelToDrive(model,"baseRandomForest.pkl")

    scoreVHyperparameterDictionary = appendToHyperparameterDictionary(scoreVsHyperparameterDictionary, criterion, nEstimators, maxDepth, maxFeatures, trainScore, devScore)
    if (iteration == 49):
      df = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary)
      display(df)
      dataPrep.saveCSVToDrive(df,"BestHyperparameters.csv")
      if (modelExistsInDrive("baseRandomForest.pkl")):
        print('best model found')
      else:
        print('best model not found')
      return

    if (iteration % 10) == 9:
      bestFiveHyperparameterGroups = createSortedScoreVsHyperparameterDF(scoreVsHyperparameterDictionary).head(5)
      display(bestFiveHyperparameterGroups)
      possibleNEstimators = adjustRange(bestFiveHyperparameterGroups['nEstimators'])
      possibleMaxDepth = adjustRange(bestFiveHyperparameterGroups['maxDepth'])
      possibleMaxFeatures = adjustRange(bestFiveHyperparameterGroups['maxFeatures'])
      if (modelExistsInDrive("baseRandomForest.pkl")):
        print('best model found')
        testSet = dataPrep.retrieveCSVFromDrive("SalesReinforcerTest.csv")
        bestModel = retrieveModelFromDrive("baseRandomForest.pkl")
        test_predictions = bestModel.predict(testSet.drop(columns=['isSubscribed']))
        testScore = f1_score(testSet['isSubscribed'], test_predictions)
        print(f"test score: {testScore}")
        return
      scoreVsHyperparameterDictionary = {
          "criterion":[],
          "nEstimators":[],
          "maxDepth":[],
          "maxFeatures":[],
          "trainScore":[],
          "devScore":[]
      }




In [13]:
if __name__ == "__main__":
  main()

iteration: 1 of 50
iteration: 2 of 50
iteration: 3 of 50
iteration: 4 of 50
iteration: 5 of 50
iteration: 6 of 50
iteration: 7 of 50
iteration: 8 of 50
iteration: 9 of 50
iteration: 10 of 50


,criterion,nEstimators,maxDepth,maxFeatures,trainScore,devScore
1,log_loss,49,95,15,1.0,0.999719
3,entropy,19,78,68,1.0,0.999157
5,log_loss,17,163,57,1.0,0.999157
6,gini,81,164,124,1.0,0.998875
7,log_loss,16,156,130,1.0,0.998875


best model found
test score: 0.9991043439319302
